In [7]:
from qiskit import QuantumCircuit, transpile, QuantumRegister, ClassicalRegister
import numpy as np
from qiskit_aer import AerSimulator
from qiskit.visualization import circuit_drawer

""" This is a controlled ADD/SUB that performs a,b -> a, a-b"""
#NOTE this version costs n CNOTs more that the b-a version, as a must be un-inverted

def ctrl_ADDSUB(qc, a, b, ctrl, c): #THIS DOES NOT ADD AND REMOVE ANCILLAS
    """
    Takes in a quantum circuit and two quantum registers (a and b),
    performs the addition, and returns the updated circuit.
    
    Parameters:
    - qc: QuantumCircuit on which the addition is performed
    - a: QuantumRegister for the first number
    - b: QuantumRegister for the second number
    - carry: QuantumRegister for the carry bits
    
    Returns:
    - QuantumCircuit: The updated quantum circuit with the addition operation applied.
    """

    n = len(a)  # number of qubits per register

    #This is if want b -> a - b
    #b flip for subtraction
    for i in b:
        qc.cx(ctrl, i)
    
    qc.cx(ctrl,a[0])
    qc.cx(ctrl,b[0])
    qc.ccx(a[0], b[0], c[0]) 
    qc.cx(ctrl,c[0])

    #apply repeated segments
    for i in range(1,n-1):
        qc.cx(c[i-1],a[i])
        qc.cx(c[i-1],b[i])
        qc.ccx(a[i], b[i], c[i])
        qc.cx(c[i-1],c[i])
        
    #end segment (we have no carry-out in this implementation)
    qc.cx(c[n-2],b[n-1]) # n-1 as e.g. n = 4 final position is index=3

    #un compute ancillas
    for i in range(n-2,0,-1): # i goes from n-2 to 1
        qc.cx(c[i-1],c[i])
        qc.ccx(a[i], b[i], c[i])
        qc.cx(c[i-1],a[i])

    #uncompute c0 and complete the operation
    qc.cx(ctrl,c[0])
    qc.ccx(a[0], b[0], c[0])
    qc.cx(ctrl,a[0])

    for i in range(0,n):
        qc.cx(a[i],b[i])
    
    return qc

""" a - b """

# Number of bits
n = 4

ctrl = QuantumRegister(1, 'ctrl')
a = QuantumRegister(n, 'a')
b = QuantumRegister(n, 'b')
carry = QuantumRegister(n-1, 'carry')

# Define a classical register with 2n bits (n for a and n for b)
result = ClassicalRegister(2 * n, 'result')

qc = QuantumCircuit(a, b, ctrl, result, carry)

# Initialize the ctrl
SUB = True

if SUB == True:
    qc.x(ctrl[0]) #control 0 - add, 1 - sub
    
print(ctrl)
#first I will make just a controlled adder so keep on 1

# Let's add a = 7 (0111) and b = 5 (0101)  

qc.x(a[0])  # a0 to 1
qc.x(b[0])  # b0 to 1

qc.x(a[1])  # a1 to 1
#qc.x(b[1])  # b1 to 1

qc.x(a[2])  # a2 to 1
qc.x(b[2])  # b2 to 1

#qc.x(a[3])  # a3 to 1
#qc.x(b[3])  # b3 to 1

#apply the ctrl add/sub operation
qc = ctrl_ADDSUB(qc, a, b, ctrl, carry)

# Measure the sum qubits and carry qubit
#qc.measure_all()

# Measure the qubits into the classical register
qc.measure(a, result[:n])      # Measure 'a' into the first n classical bits
qc.measure(b, result[n:2*n])   # Measure 'b' into the last n classical bits


# For execution
simulator = AerSimulator()
compiled_circuit = transpile(qc, simulator)
sim_result = simulator.run(compiled_circuit).result()
counts = sim_result.get_counts()

print(counts)

print("Measurement results:")
for bitstring, count in counts.items():
    # bitstring is of the form 'result' where the first n bits are for 'a' and the last n bits are for 'b'
    a_result = bitstring[-n:]  # Last n bits for b
    b_result = bitstring[-2*n:-n]  # First n bits for a
    a_decimal_value = int(a_result, 2)
    
    # Get the correct decimal value for b
    if SUB == True and b_result[0] == '1':
        b_decimal_value = -1 * (2**(n-1))
        for i in range(1, n):
            b_decimal_value += int(b_result[i]) * (2**(n-1-i))
    else:
        b_decimal_value = int(b_result, 2)
        
    print(f"a = {a_result} (decimal: {a_decimal_value}), b = {b_result} (decimal: {b_decimal_value})")

# Draw the circuit
qc_drawn = circuit_drawer(qc, output='text')
print(qc_drawn)


QuantumRegister(1, 'ctrl')
{'00100111': 1024}
Measurement results:
a = 0111 (decimal: 7), b = 0010 (decimal: 2)
          ┌───┐                    ┌───┐                                   »
     a_0: ┤ X ├────────────────────┤ X ├───────■───────────────────────────»
          ├───┤                    └─┬─┘       │       ┌───┐               »
     a_1: ┤ X ├──────────────────────┼─────────┼───────┤ X ├───────■───────»
          ├───┤                      │         │       └─┬─┘       │       »
     a_2: ┤ X ├──────────────────────┼─────────┼─────────┼─────────┼───────»
          └───┘                      │         │         │         │       »
     a_3: ───────────────────────────┼─────────┼─────────┼─────────┼───────»
          ┌───┐┌───┐                 │  ┌───┐  │         │         │       »
     b_0: ┤ X ├┤ X ├─────────────────┼──┤ X ├──■─────────┼─────────┼───────»
          └───┘└─┬─┘┌───┐            │  └─┬─┘  │         │  ┌───┐  │       »
     b_1: ───────┼──┤ X ├────────────┼───